# Estudo de temperatura — llama3.1:8b, BFI-44 (mesmo desenho do Experimento A)

Pedido do orientador: verificar como os resultados se comportam com **temperatura menor**.

- O Experimento A usou **temperatura 0,7**. Aqui rodamos o **mesmo desenho** (5 traços simultâneos, BFI-44, IPA, N=300, `--seed 123`) com **0,0** e **0,3**.
- Como a seed é a mesma, os **300 perfis de personalidade são idênticos** aos do Experimento A: a única diferença entre as rodadas é a temperatura. Isso permite comparar agente a agente.
- Temperatura 0,0 = resposta determinística (o modelo sempre escolhe a palavra mais provável). Temperatura 0,3 = intermediária.

## Antes de rodar

1. `Ambiente de execução → Alterar tipo → GPU T4`.
2. Atualize no Drive a pasta `projeto-tcc` com a versão **nova** do `src/geracao/gerar_agentes.py` (que aceita `--temperatura`) e com `resultados/principal/exp_a_bfi44_multitraco_8b.csv` (para a comparação no final).

## Fluxo

1. GPU → instala Ollama → baixa `llama3.1:8b`
2. Copia o projeto do Drive
3. Roda (ou retoma) a temperatura 0,0, depois a 0,3. **Pode rodar a mesma célula de novo se a sessão cair: ela continua de onde parou, sem duplicar perfis.**
4. Salva no Drive e compara com o Experimento A (0,7)

Cada rodada gera um `.csv` e um `.log` com o andamento e resumos parciais a cada 10 agentes.

---
## Passo 1 — GPU e Ollama

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU disponível:', r.stdout.strip()) if r.returncode == 0 else print('ATENÇÃO: GPU não detectada (vai rodar em CPU, mais lento).')

In [ ]:
!apt-get install -y -q zstd
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
import subprocess, time, os, requests
OLLAMA_BIN = '/usr/local/bin/ollama'

def garantir_ollama_rodando():
    try:
        if requests.get('http://localhost:11434', timeout=5).ok:
            return
    except Exception:
        pass
    print('Iniciando o servidor do Ollama...')
    subprocess.Popen([OLLAMA_BIN, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    time.sleep(5)

def garantir_modelo(modelo='llama3.1:8b'):
    r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
    if modelo not in r.stdout:
        print(f'Baixando {modelo} (alguns minutos)...')
        subprocess.run([OLLAMA_BIN, 'pull', modelo], check=True)

garantir_ollama_rodando()
garantir_modelo()
print(subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True).stdout)

---
## Passo 2 — Copiar o projeto do Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_PROJETO = '/content/drive/MyDrive/tcc/projeto-tcc'   # ajuste se a pasta estiver em outro lugar
LOCAL_PROJETO = '/content/projeto-tcc'

if not os.path.exists(DRIVE_PROJETO):
    raise FileNotFoundError(f'Pasta não encontrada: {DRIVE_PROJETO}')
if os.path.exists(LOCAL_PROJETO):
    shutil.rmtree(LOCAL_PROJETO)
shutil.copytree(DRIVE_PROJETO, LOCAL_PROJETO)
os.makedirs(f'{LOCAL_PROJETO}/resultados/temperatura', exist_ok=True)

# Traz de volta do Drive rodadas já começadas (para retomar depois de uma queda)
for arq in ['exp_a_t00_bfi44_8b.csv', 'exp_a_t00_bfi44_8b.log', 'exp_a_t03_bfi44_8b.csv', 'exp_a_t03_bfi44_8b.log']:
    origem = f'{DRIVE_PROJETO}/resultados/temperatura/{arq}'
    if os.path.exists(origem):
        shutil.copy2(origem, f'{LOCAL_PROJETO}/resultados/temperatura/{arq}')
        print('Retomando a partir de', origem)

if '--temperatura' not in open(f'{LOCAL_PROJETO}/src/geracao/gerar_agentes.py', encoding='utf-8').read():
    raise RuntimeError('O gerar_agentes.py do Drive é a versão antiga (sem --temperatura). Atualize a pasta no Drive.')
print('Projeto pronto em', LOCAL_PROJETO)

---
## Passo 3 — Rodar (ou retomar) cada temperatura

A função abaixo calcula quantos agentes faltam para chegar a 300 e roda só esses. Depois de cada rodada, copia o CSV e o log para o Drive.

In [ ]:
import csv, shutil

N_TOTAL = 300

def rodar_temperatura(temp, rotulo):
    garantir_ollama_rodando()
    garantir_modelo()
    saida = f'resultados/temperatura/exp_a_{rotulo}_bfi44_8b.csv'
    caminho = f'{LOCAL_PROJETO}/{saida}'
    feitos = 0
    if os.path.exists(caminho):
        with open(caminho, encoding='utf-8') as f:
            feitos = max(0, sum(1 for _ in f) - 1)
    faltam = N_TOTAL - feitos
    print(f'Temperatura {temp}: {feitos} já gerados, faltam {faltam}.')
    if faltam > 0:
        env = {**os.environ, 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}
        cmd = ['python', 'src/geracao/gerar_agentes.py', '--n', str(faltam), '--output', saida,
               '--modo', 'multi', '--seed', '123', '--modelo', 'llama3.1:8b', '--temperatura', str(temp)]
        proc = subprocess.Popen(cmd, cwd=LOCAL_PROJETO, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        for linha in proc.stdout:
            print(linha, end='')
        proc.wait()
    destino = f'{DRIVE_PROJETO}/resultados/temperatura'
    os.makedirs(destino, exist_ok=True)
    for ext in ('.csv', '.log'):
        arq = caminho.replace('.csv', ext)
        if os.path.exists(arq):
            shutil.copy2(arq, destino)
            print('Salvo no Drive:', f'{destino}/{os.path.basename(arq)}')

In [ ]:
rodar_temperatura(0.0, 't00')

In [ ]:
rodar_temperatura(0.3, 't03')

---
## Passo 4 — Comparação com o Experimento A (temperatura 0,7)

Esta é só uma conferência rápida; a análise completa roda localmente com `src/analise/analise_temperatura_modelos.py`.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from scipy import stats

DIMS = ['Extroversão', 'Afabilidade', 'Consciência', 'Neuroticismo', 'Abertura à Experiência']

def carregar(caminho):
    df = pd.read_csv(caminho)
    df = df[df['ipa_perfil'].isin(['Conservador', 'Moderado', 'Arrojado'])].copy()
    df['ipa_score_sem_q1'] = df['ipa_score'] - df['ipa_q1_atribuida'].map({'a': 4, 'b': 3, 'c': 2, 'd': 1})
    return df

rodadas = {
    '0,0': f'{LOCAL_PROJETO}/resultados/temperatura/exp_a_t00_bfi44_8b.csv',
    '0,3': f'{LOCAL_PROJETO}/resultados/temperatura/exp_a_t03_bfi44_8b.csv',
    '0,7 (Exp. A)': f'{LOCAL_PROJETO}/resultados/principal/exp_a_bfi44_multitraco_8b.csv',
}
for nome, caminho in rodadas.items():
    if not os.path.exists(caminho):
        print(nome, ': ainda não gerado'); continue
    df = carregar(caminho)
    X = sm.add_constant(df[[f'nivel_{d}_num' for d in DIMS]])
    m = sm.OLS(df['ipa_score_sem_q1'], X).fit()
    print(f'=== Temperatura {nome}: N válido={len(df)}, IPA média={df.ipa_score_sem_q1.mean():.2f} dp={df.ipa_score_sem_q1.std():.2f}, R²={m.rsquared:.3f}')
    for d in DIMS:
        rm = stats.spearmanr(df[f'nivel_{d}_num'], df[f'bfi_{d}'])[0]
        print(f'   {d:<24} manipulação ρ={rm:+.3f} | OLS coef={m.params[f"nivel_{d}_num"]:+.3f} p={m.pvalues[f"nivel_{d}_num"]:.4f}')